### Notebook
- **Name:** `cs_wn.ipynb`
- **Created/updated:** 2026-02-27
- **Python:** 3.x

### Purpose
Run/organize WindNinja-based wind estimation workflow for the corridor.

### Inputs
WindNinja outputs and project configuration.

### Outputs
WindNinja run artifacts and standardized outputs for subsequent notebooks.

### Dependencies
- (see imports below)

### Usage
Executed by the project pipeline (e.g., via Papermill) or run interactively in Jupyter.

### Notes
- Keep paths and parameters centralized in `config.toml` / `CONFIG_PATH` where applicable.


In [ ]:
# Parameters (Papermill)
CONFIG_PATH = "" # r"E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml" # "config.toml"
PROJECT_ROOT = r"E:\mario\python\criticalspam"

### WindNinja

Wind estimation along the conductor corridor using WindNinja outputs.


### Packages


In [ ]:
import subprocess,time,os 
import sys


from pathlib import Path
from typing import Optional

import tomllib

import time, subprocess

from dataclasses import dataclass

In [ ]:
# Ruta al directorio raíz del proyecto (sube 1 nivel desde notebooks/)
#ROOT = Path.cwd().resolve().parent

# Añade src/ al path de importación
SRC = Path(PROJECT_ROOT + r"\src") 
print(f"Adding to sys.path: {SRC}")

#SRC =r'E:\mario\python\criticalspam\src'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ya puedes importar
from cs_utils import Config, join_base, load_config_toml
from   ninja_utils import delete_format_files,get_dates_commands

In [ ]:
#CONFIG_PATH = "E:\\mario\\trabajos2\\viesgo_edp_portugal_cic\\estudios_microclimaticos\\Corredoria_Grado_1_y_2\\config.toml"

#config_toml_path = CONFIG_PATH
cfg = load_config_toml(CONFIG_PATH)


print("cs_wn.ipynb")
print("Config path:", CONFIG_PATH)

### Set environment


In [ ]:
base = Path( cfg.general_path )

wind_ninja_exe      = r"C:\WindNinja\WindNinja-3.12.1\bin\WindNinja_cli"
wx_station_filename = join_base(base, cfg.in_weather_file) #os.path.join("WindNinja_PointInit_Path.csv")
elevation_file      = join_base(base, cfg.out_mdt_tif) #os.path.join("files_DEMs","MDT25-ETRS89-H29-COB2-merged_CUT3.tif")
path_output         = join_base(base, cfg.out_wn) #os.path.join("files_out")
config_file         = os.path.join("config.cfg")

In [ ]:
#print("wx_station_filename:", wx_station_filename)
#print("elevation_file:", elevation_file)
#print("path_output:", path_output)

In [ ]:
wx_station_filename = str(wx_station_filename)
elevation_file      = str(elevation_file)
path_output         = str(path_output)
print("wx_station_filename:", wx_station_filename)
print("elevation_file:", elevation_file)
print("path_output:", path_output)

In [ ]:
type(wx_station_filename)

In [ ]:
# dates_commands .- dictionary with information about dates and number of simulation.
# dates_correct .- number of available time simulations
dates_commands, dates_correct = get_dates_commands( wx_station_filename )

print("Número de fechas correctas en el archivo de estaciones:", dates_correct)
if dates_correct:
    #raise ValueError("No hay fechas correctas en el archivo de estaciones.")
    dates_commands

In [ ]:
if dates_correct:
    
    print("--------- Ejecutando WindNinja ---------")
    mesh_resolution   = cfg.mesh_resolution # spatial resolution [m]
    num_threads       = cfg.num_threads
    number_time_steps = False  # if false number_time_steps are computed as len(dates)-2
    #number_time_steps = 3
     
    if number_time_steps:
       dates_commands["--number_time_steps"] = number_time_steps
    else:
       number_time_steps = dates_commands["--number_time_steps"]
    
    # DATES AND TIME STEP
    list_dates_commands = []
    for di in dates_commands:
        list_dates_commands.append(di)
        list_dates_commands.append(f"{dates_commands[di]:02d}")
        print(f"{di} {dates_commands[di]:02d}")


    print("--------- Antes de try: ")
    print("wx_station_filename:", wx_station_filename)


    start_time = time.time()
    commands   = [wind_ninja_exe,"--elevation_file",elevation_file,
                                 "--wx_station_filename",wx_station_filename,
                                 "--output_path",path_output,
                                 "--config_file",config_file,
                                 "--time_zone","Europe/London",
                                 "--mesh_resolution",f'{mesh_resolution}',
                                 "--num_threads",f'{num_threads}',
                                 *list_dates_commands]
    try:
        #commands = ['C:\\WindNinja\\WindNinja-3.12.1\\bin\\WindNinja_cli', '--elevation_file', 'E:\\mario\\python\\criticalspam\\files_DEMs\\MDT25-ETRS89-H29-COB2-merged_CUT3.tif', '--wx_station_filename', 'WindNinja_PointInit_Path.csv', '--output_path', 'files_out', '--config_file', 'config.cfg', '--time_zone', 'Europe/London', '--mesh_resolution', '100', '--num_threads', '8', '--number_time_steps', '02', '--start_year', '2025', '--start_month', '01', '--start_day', '01', '--start_hour', '00', '--start_minute', '00', '--stop_year', '2025', '--stop_month', '02', '--stop_day', '08', '--stop_hour', '07', '--stop_minute', '15']
        #result = subprocess.run(commands)

        # 1) Asegura que el directorio existe
        out_dir = Path(path_output)
        out_dir.mkdir(parents=True, exist_ok=True)

        # 2) Snapshot antes
        before = {p.resolve() for p in out_dir.glob("**/*") if p.is_file()}

        t0 = time.time()

        # 3) Ejecuta capturando salida (si hay avisos/errores los verás)
        res = subprocess.run(commands, capture_output=True, text=True)

        dt = time.time() - t0
        print("Return code:", res.returncode)
        print("Elapsed [s]:", dt)

        if res.stdout:
            print("\n--- STDOUT (tail) ---\n", res.stdout[-2000:])
        if res.stderr:
            print("\n--- STDERR (tail) ---\n", res.stderr[-2000:])

        # 4) Snapshot después
        after = [p for p in out_dir.glob("**/*") if p.is_file()]
        new_files = [p for p in after if p.resolve() not in before]

        print("\nNuevos ficheros en output_path:", len(new_files))
        for p in sorted(new_files, key=lambda x: x.stat().st_mtime, reverse=True)[:20]:
            print("  ", p)

        # 5) Si no aparece nada, mira también la carpeta del DEM (salida por defecto si no redirige)
        dem_dir = Path(elevation_file).parent
        dem_out = [p for p in dem_dir.glob("*") if p.is_file()]
        print("\nCarpeta DEM:", dem_dir)
        print("Ficheros (muestra):", [p.name for p in dem_out[:20]])

        # IMPORTANTE: deja desactivado el borrado hasta verificar salidas reales
        # time.sleep(3); delete_format_files(path_output)

        
    except subprocess.CalledProcessError as e:
        print(f"Error: WindNinja command failed with exit code {e.returncode}")
        print(f"Error output: {e.stderr}")
      
    elapsed_time = time.time() - start_time




    
print(' ')
print('Print Results')
print('#'*80)
print(f'Mesh_resolution: {mesh_resolution}')
print(f'Num_threads: {num_threads}')
print(f'Mumber_time_steps: {number_time_steps}')

print(' ')
print(f'Tiempo Total: {elapsed_time} s')
print(f'Tiempo Sim_i: {elapsed_time/number_time_steps} s')
print(' ')

# delete files
################################  
time.sleep(3)
#;delete_format_files(path_output)
    
print(" ")
print("All Commands")
for i in range(1,len(commands)-1,2):
    print(f' {commands[i]}: {commands[i+1]}' )
print(" ")
print(commands)
print(" ")
